In [1]:
%pip install crewai langchain langchain-openai langchain-community langchain-tavily tavily-python pydantic

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install litellm
%pip install -U crewai

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import requests
import litellm
from crewai.llm import LLM
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool
from dotenv import load_dotenv

load_dotenv()

# Configure LiteLLM
litellm.drop_params = True
# Monkey patch litellm.completion to handle parameter mapping
original_completion = litellm.completion

def patched_completion(*args, **kwargs):
    # If max_tokens is present and max_completion_tokens is not, map it
    if 'max_tokens' in kwargs and 'max_completion_tokens' not in kwargs:
        kwargs['max_completion_tokens'] = kwargs.pop('max_tokens')
    return original_completion(*args, **kwargs)

# Apply the patch
litellm.completion = patched_completion

In [4]:
# Tavily API Key
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [ ]:
# ---- Custom CrewAI Tool for Web Search ----
class TavilySearchTool(BaseTool):
    name: str = "web_search"
    description: str = "Search the web for recent information."

    def _run(self, query: str):
        url = "https://api.tavily.com/search"
        payload = {
            "api_key": TAVILY_API_KEY,
            "query": query,
            "max_results": 3
        }

        response = requests.post(url, json=payload, timeout=30)
        response.raise_for_status()
        data = response.json()

        results = []
        for r in data.get("results", []):
            title = r.get("title", "No title")
            link = r.get("url", "No URL")
            results.append(f"{title} - {link}")

        return "\n".join(results) if results else "No web results found."


search_tool = TavilySearchTool()

# ---- Azure LLM - FIXED ----
# Using max_tokens (not max_completion_tokens) with the monkey patch
llm = LLM(
    model=f"azure/{os.getenv('AZURE_OPENAI_CHAT_DEPLOYMENT')}",
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    base_url=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-12-01-preview"),
    is_litellm=True,
    temperature=1, # some models don't support temperature
    max_tokens=3500  # This will be converted to max_completion_tokens
)

In [6]:
# --------------------------------------------------
# Agents
# --------------------------------------------------
researcher = Agent(
    role="AI Researcher",
    goal="Find the latest advancements in AI for healthcare",
    backstory=(
        "You are an expert in artificial intelligence and stay updated "
        "with the latest research trends in healthcare."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=2,
    tools=[search_tool]
)

writer = Agent(
    role="Technical Writer",
    goal="Summarize research into an executive report",
    backstory=(
        "You are an experienced technical writer with expertise in "
        "summarizing healthcare research for executives."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm
)

In [7]:
# --------------------------------------------------
# SERIAL EXECUTION
# --------------------------------------------------
task_research = Task(
    description=(
        "Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4 sentences."
        "Do not call tools again after getting results."
    ),
    expected_output=(
        "Detailed notes on three advancements, with names and explanations."
    ),
    agent=researcher
)

task_write = Task(
    description=(
        "Write a short executive summary using the research notes provided by the AI Researcher. "
        "Limit the answer to about 100 words."
    ),
    expected_output=(
        "An executive summary report of the top 3 AI advancements in healthcare."
    ),
    agent=writer,
    context=[task_research]
)

print("\n=== SERIAL EXECUTION ===")

crew_serial = Crew(
    agents=[researcher, writer],
    tasks=[task_research, task_write],
    verbose=True
)

serial_result = await crew_serial.kickoff_async()

print("\n[Serial Result]:\n")
try:
    print(serial_result.raw)
except AttributeError:
    print(serial_result)


=== SERIAL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 63da0a74-f32d-410e-bd9e-51c8d7b6ac88                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│  ID: d8762f83-b1e3-4282-99f5-1ccc47953666                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use the web search results to explain the top 3 recent advancements in AI for healthcare in 3-4          │
│  sentences.Do not call tools again after getting results.                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Executive summary—Top 3 AI advancements in healthcare                                                          │
│                                                                                                                 │
│  1) Clinical LLMs: Foundation models fine‑tuned for EHR summarization, decision support, triage and report      │
│  generation produce context‑aware outputs that reduce clinician documentation burden in trials; deployments     │
│  use retrieval‑augmented generation and safety layers; pilots validate guardrails.                              │
│                                                                                                                 │
│  2) Multimodal models & imaging: Systems combining CT/MRI/X‑ray, text and genomics reach or exceed clinician    │
│  performance for cancer detection, fracture ID and segmentation in multi‑center studies; integrated modalities  │
│  enable outcome and treatment‑response prediction with explainability.                                          │
│                                                                                                                 │
│  3) AI‑driven drug discovery: Generative and structure‑aware models (post‑AlphaFold) accelerate lead design,    │
│  binding evaluation and ADMET triage, shortening timelines and speeding progression to validated preclinical    │
│  hits.                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short executive summary using the research notes provided by the AI Researcher. Limit the        │
│  answer to about 100 words.                                                                                     │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 63da0a74-f32d-410e-bd9e-51c8d7b6ac88                                                                       │
│  Final Output: Executive summary—Top 3 AI advancements in healthcare                                            │
│                                                                                                                 │
│  1) Clinical LLMs: Foundation models fine‑tuned for EHR summarization, decision support, triage and report      │
│  generation produce context‑aware outputs that reduce clinician documentation burden in trials; deployments     │
│  use retrieval‑augmented generation and safety layers; pilots validate guardrails.                              │
│                                                                                                                 │
│  2) Multimodal models & imaging: Systems combining CT/MRI/X‑ray, text and genomics reach or exceed clinician    │
│  performance for cancer detection, fracture ID and segmentation in multi‑center studies; integrated modalities  │
│  enable outcome and treatment‑response prediction with explainability.                                          │
│                                                                                                                 │
│  3) AI‑driven drug discovery: Generative and structure‑aware models (post‑AlphaFold) accelerate lead design,    │
│  binding evaluation and ADMET triage, shortening timelines and speeding progression to validated preclinical    │
│  hits.                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Serial Result]:

Executive summary—Top 3 AI advancements in healthcare

1) Clinical LLMs: Foundation models fine‑tuned for EHR summarization, decision support, triage and report generation produce context‑aware outputs that reduce clinician documentation burden in trials; deployments use retrieval‑augmented generation and safety layers; pilots validate guardrails.

2) Multimodal models & imaging: Systems combining CT/MRI/X‑ray, text and genomics reach or exceed clinician performance for cancer detection, fracture ID and segmentation in multi‑center studies; integrated modalities enable outcome and treatment‑response prediction with explainability.

3) AI‑driven drug discovery: Generative and structure‑aware models (post‑AlphaFold) accelerate lead design, binding evaluation and ADMET triage, shortening timelines and speeding progression to validated preclinical hits.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# --------------------------------------------------
# PARALLEL EXECUTION
# --------------------------------------------------
task_parallel_1 = Task(
    description=(
        "Use web search to list 5 AI companies focusing on drug discovery. "
        "For each company, give one short line about what they specialize in."
    ),
    expected_output="Company names and what they specialize in.",
    async_execution=True,
    agent=researcher
)

task_parallel_2 = Task(
    description=(
        "Write a short report on how AI is transforming patient diagnostics. "
        "Limit the answer to about 100 words."
    ),
    expected_output="A short report with examples and explanation.",
    agent=writer
)

print("\n=== PARALLEL EXECUTION ===")

crew_parallel = Crew(
    agents=[researcher, writer],
    tasks=[task_parallel_1, task_parallel_2],
    verbose=True
)

parallel_result = await crew_parallel.kickoff_async()

print("\n[Parallel Result]:\n")
try:
    print(parallel_result.raw)
except AttributeError:
    print(parallel_result)


=== PARALLEL EXECUTION ===


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7f95b026-51cf-465e-8358-b921914e2e0d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│  ID: 029f42ca-1562-4c7e-9d1c-02ebb0686702                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: AI Researcher                                                                                           │
│                                                                                                                 │
│  Task: Use web search to list 5 AI companies focusing on drug discovery. For each company, give one short line  │
│  about what they specialize in.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: web_search                                                                                               │
│  Args: {"query": "Atomwise drug discovery what they do specialization"}                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  AI is accelerating patient diagnostics by integrating multimodal data, improving pattern recognition, and      │
│  shortening biomarker discovery cycles. Examples: companies like Recursion use high‑throughput biology plus     │
│  machine learning to map disease phenotypes; BenevolentAI’s knowledge‑graph approaches help surface biomarker   │
│  and disease‑link hypotheses; Insilico, Exscientia, and Atomwise apply generative and structure‑driven models   │
│  to prioritize molecular signatures and design targeted assays. Resulting gains include faster, earlier, and    │
│  more specific diagnoses, improved stratification for precision therapies, and streamlined lab workflows.       │
│  Implementation requires clinical validation, interoperability, and regulatory oversight to manage bias,        │
│  safety, and adoption risks.                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Write a short report on how AI is transforming patient diagnostics. Limit the answer to about 100        │
│  words.                                                                                                         │
│  Agent: Technical Writer                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7f95b026-51cf-465e-8358-b921914e2e0d                                                                       │
│  Final Output: AI is accelerating patient diagnostics by integrating multimodal data, improving pattern         │
│  recognition, and shortening biomarker discovery cycles. Examples: companies like Recursion use                 │
│  high‑throughput biology plus machine learning to map disease phenotypes; BenevolentAI’s knowledge‑graph        │
│  approaches help surface biomarker and disease‑link hypotheses; Insilico, Exscientia, and Atomwise apply        │
│  generative and structure‑driven models to prioritize molecular signatures and design targeted assays.          │
│  Resulting gains include faster, earlier, and more specific diagnoses, improved stratification for precision    │
│  therapies, and streamlined lab workflows. Implementation requires clinical validation, interoperability, and   │
│  regulatory oversight to manage bias, safety, and adoption risks.                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[Parallel Result]:

AI is accelerating patient diagnostics by integrating multimodal data, improving pattern recognition, and shortening biomarker discovery cycles. Examples: companies like Recursion use high‑throughput biology plus machine learning to map disease phenotypes; BenevolentAI’s knowledge‑graph approaches help surface biomarker and disease‑link hypotheses; Insilico, Exscientia, and Atomwise apply generative and structure‑driven models to prioritize molecular signatures and design targeted assays. Resulting gains include faster, earlier, and more specific diagnoses, improved stratification for precision therapies, and streamlined lab workflows. Implementation requires clinical validation, interoperability, and regulatory oversight to manage bias, safety, and adoption risks.


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯